# 🌤️ Weather Forecasting MLOps — Model Training
**Train Prophet + LSTM models on Google Colab**

Pipeline:
1. Clone repo + Install dependencies
2. Fetch weather data (Open-Meteo API → SQLite)
3. Train Prophet (6 cities × hourly + daily)
4. Train LSTM (hourly single-city + daily multi-city)
5. Evaluate & push models back to GitHub

## 1️⃣ Setup — Clone repo & Install

In [ ]:
# Clone repo
!git clone https://github.com/LinhThaoPham/weather-forecasting-mlop.git
%cd weather-forecasting-mlop

# Install dependencies
!pip install -q prophet tensorflow keras pandas numpy scikit-learn statsmodels joblib requests python-dotenv

In [ ]:
# Verify imports
import sys, os
sys.path.insert(0, os.getcwd())

import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

## 2️⃣ Fetch Weather Data → SQLite

In [ ]:
from src.config.db import init_db
from src.data_pipeline.store_data import seed_all_cities

# Init DB + Fetch 2 years of data for 6 cities
# This takes ~5 minutes (API rate limiting)
seed_all_cities(days=730, delay=0.5)

In [ ]:
# Verify DB stats
from src.config.db import get_connection
with get_connection() as conn:
    for table in ['weather_historical', 'weather_forecast', 'weather_current']:
        count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
        print(f'{table}: {count:,} rows')

## 3️⃣ Train Prophet Models (6 cities × 2 modes)

In [ ]:
import json
import pandas as pd
from src.config.cities import CITIES
from src.config.constants import TRAIN_SPLIT_RATIO, model_filename
from src.data_pipeline.feature_engineering import add_features
from src.models_logic.prophet_model import train_prophet, save_prophet
from src.training.evaluate import evaluate_prophet

CITY_IDS = sorted(CITIES.keys())
TRAIN_DIR = 'models/training_temp'
os.makedirs(TRAIN_DIR, exist_ok=True)

prophet_metrics = {}

def fetch_training_data(city_id):
    with get_connection() as conn:
        rows = conn.execute(
            '''SELECT timestamp, temperature, humidity, cloud_cover
               FROM weather_historical WHERE city_id = ? ORDER BY timestamp''',
            (city_id,)).fetchall()
    df = pd.DataFrame(rows, columns=['ds', 'y', 'humidity', 'cloud_cover'])
    df['ds'] = pd.to_datetime(df['ds'])
    return df.dropna(subset=['y'])

for city_id in CITY_IDS:
    print(f'\n=== Prophet: {city_id} ===')
    data = fetch_training_data(city_id)
    print(f'  Data: {len(data):,} rows')

    # Hourly
    hourly = add_features(data.copy(), is_hourly=True)
    split = int(len(hourly) * TRAIN_SPLIT_RATIO)
    model_h = train_prophet(hourly.iloc[:split], is_hourly=True)
    save_prophet(model_h, os.path.join(TRAIN_DIR, model_filename('prophet', 'hourly', city_id)))
    m = evaluate_prophet(model_h, hourly.iloc[split:], 'hourly')
    prophet_metrics[f'prophet_hourly_{city_id}'] = m
    print(f'  Hourly MAE: {m["mae"]}°C')

    # Daily
    daily = add_features(data.copy(), is_hourly=False)
    split = int(len(daily) * TRAIN_SPLIT_RATIO)
    model_d = train_prophet(daily.iloc[:split], is_hourly=False)
    save_prophet(model_d, os.path.join(TRAIN_DIR, model_filename('prophet', 'daily', city_id)))
    m = evaluate_prophet(model_d, daily.iloc[split:], 'daily')
    prophet_metrics[f'prophet_daily_{city_id}'] = m
    print(f'  Daily MAE: {m["mae"]}°C')

print('\n✅ Prophet training complete!')
print(json.dumps(prophet_metrics, indent=2))

## 4️⃣ Train LSTM Models

In [ ]:
import numpy as np
import joblib
from src.config.constants import *
from src.data_pipeline.feature_engineering import (
    normalize_data, normalize_multi_city, sliding_window, build_multi_city_daily
)
from src.models_logic.lstm_model import create_lstm_model
from src.training.evaluate import evaluate_lstm, evaluate_lstm_multi_city

lstm_metrics = {}

# ── 4a. LSTM Hourly (single-city: Hanoi) ──
print('=== LSTM Hourly (Hanoi) ===')
hanoi_data = fetch_training_data('hanoi')
hourly_norm, hourly_scaler = normalize_data(hanoi_data['y'].values)
X, y = sliding_window(hourly_norm, HOURLY_LOOKBACK, HOURLY_HORIZON)
X = X.reshape((X.shape[0], X.shape[1], 1))
y = y.reshape((y.shape[0], y.shape[1], 1))

split = int(len(X) * TRAIN_SPLIT_RATIO)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
print(f'  Train: {X_train.shape}, Val: {X_val.shape}')

lstm_h = create_lstm_model(HOURLY_LOOKBACK, HOURLY_HORIZON, feature_dim=1)
lstm_h.model.summary()
history_h = lstm_h.train(X_train, y_train, X_val, y_val, epochs=MAX_EPOCHS, batch_size=BATCH_SIZE)
lstm_h.save(os.path.join(TRAIN_DIR, model_filename('lstm', 'hourly')))
joblib.dump(hourly_scaler, os.path.join(TRAIN_DIR, scaler_filename('hourly')))

m = evaluate_lstm(lstm_h.model, X_val, y_val, hourly_scaler)
m['val_loss'] = round(float(history_h.history['val_loss'][-1]), 6)
lstm_metrics['lstm_hourly'] = m
print(f'  ✅ MAE: {m["mae"]}°C, Val Loss: {m["val_loss"]}')

In [ ]:
# ── 4b. LSTM Daily Multi-City (6 cities) ──
print('=== LSTM Daily Multi-City ===')
df_pivot, city_ids = build_multi_city_daily()
daily_data = df_pivot.values

daily_norm, daily_scaler = normalize_multi_city(daily_data)
X, y = sliding_window(daily_norm, DAILY_LOOKBACK, DAILY_HORIZON)

split = int(len(X) * TRAIN_SPLIT_RATIO)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]
print(f'  Train: {X_train.shape}, Val: {X_val.shape}')

lstm_d = create_lstm_model(DAILY_LOOKBACK, DAILY_HORIZON, feature_dim=NUM_CITIES)
lstm_d.model.summary()
history_d = lstm_d.train(X_train, y_train, X_val, y_val, epochs=MAX_EPOCHS, batch_size=BATCH_SIZE)
lstm_d.save(os.path.join(TRAIN_DIR, model_filename('lstm', 'daily')))
joblib.dump(daily_scaler, os.path.join(TRAIN_DIR, scaler_filename('daily')))

m = evaluate_lstm_multi_city(lstm_d.model, X_val, y_val, daily_scaler)
m['val_loss'] = round(float(history_d.history['val_loss'][-1]), 6)
m['city_ids'] = city_ids
lstm_metrics['lstm_daily'] = m
print(f'  ✅ MAE: {m["mae"]}°C, Val Loss: {m["val_loss"]}')

## 5️⃣ Visualize Training History

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hourly
axes[0].plot(history_h.history['loss'], label='Train Loss')
axes[0].plot(history_h.history['val_loss'], label='Val Loss')
axes[0].set_title('LSTM Hourly — Training History')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE Loss')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Daily
axes[1].plot(history_d.history['loss'], label='Train Loss')
axes[1].plot(history_d.history['val_loss'], label='Val Loss')
axes[1].set_title('LSTM Daily Multi-City — Training History')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## 6️⃣ Deploy Models → models/current/

In [ ]:
import shutil

CURRENT_DIR = 'models/current'
os.makedirs(CURRENT_DIR, exist_ok=True)

# Copy all trained models to current/
count = 0
for f in os.listdir(TRAIN_DIR):
    shutil.copy2(os.path.join(TRAIN_DIR, f), os.path.join(CURRENT_DIR, f))
    count += 1
    print(f'  Deployed: {f}')

print(f'\n✅ {count} model files deployed to {CURRENT_DIR}/')

# Update registry
from datetime import datetime
all_metrics = {**prophet_metrics, **lstm_metrics}
registry = {
    'current_version': f'v_{datetime.now().strftime("%Y-%m-%d")}',
    'models': [{
        'version': f'v_{datetime.now().strftime("%Y-%m-%d")}',
        'trained_at': datetime.now().isoformat(),
        'trained_on': 'Google Colab (GPU)',
        'metrics': all_metrics,
        'status': 'active'
    }]
}
with open('models/registry.json', 'w') as f:
    json.dump(registry, f, indent=2)
print('✅ Registry updated')

## 7️⃣ Metrics Summary

In [ ]:
print('=' * 60)
print('📊 TRAINING RESULTS SUMMARY')
print('=' * 60)
for name, m in sorted(all_metrics.items()):
    mae = m.get('mae', 'N/A')
    rmse = m.get('rmse', 'N/A')
    print(f'  {name:35s} MAE: {mae:>8}°C  RMSE: {rmse:>8}°C')
print('=' * 60)

## 8️⃣ Push Models back to GitHub
⚠️ Chạy cell này nếu muốn push models lên repo

In [ ]:
# Cấu hình Git credentials
# Thay YOUR_TOKEN bằng GitHub Personal Access Token
# Tạo token tại: https://github.com/settings/tokens

GITHUB_TOKEN = ''  # ← PASTE TOKEN VÀO ĐÂY

if GITHUB_TOKEN:
    !git config user.name 'colab-bot'
    !git config user.email 'colab@train.bot'
    !git remote set-url origin https://{GITHUB_TOKEN}@github.com/LinhThaoPham/weather-forecasting-mlop.git
    !git add models/ data/weather_forecast.db
    !git commit -m "🤖 Colab retrain — $(date +'%Y-%m-%d') | GPU trained"
    !git push origin main
    print('✅ Models pushed to GitHub!')
else:
    print('⚠️ Set GITHUB_TOKEN to push. Or download files manually below.')

## 📥 Download Models (nếu không push)

In [ ]:
# Download models as zip
!zip -r trained_models.zip models/current/ models/registry.json data/weather_forecast.db

from google.colab import files
files.download('trained_models.zip')
print('📥 Download started!')

In [ ]:
# Cleanup
shutil.rmtree(TRAIN_DIR, ignore_errors=True)
print('🧹 Cleaned up temp files')